# Foot Alignment Full Pipeline

This notebook records the reproducible steps used for the current shoe and foot alignment pipeline.

The final debug outputs from this notebook are written under:

```text
/data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/final
```

## Section 1: Turntable Canonicalization

The original processed dataset is here:

```text
/data/abelde/datasets/processed/gshell_shoes
```

The images look like a consistent turntable dataset, but COLMAP can choose a different global yaw for each shoe. That means two shoes can have the same image ordering, but their trained meshes can face different world directions.

This step fixes that by creating the processed dataset used for the current GShell training:

```text
/data/abelde/datasets/processed/gshell_shoes_turntable_canonical
```

What changes:

- `transforms.json` is rewritten.
- Each frame's `transform_matrix` is yaw-rotated.
- `img01.jpg` is forced to land at the same turntable angle for every shoe.

What does not change:

- The actual image files are not rotated.
- The mask files are not changed.
- `camera_angle_x` is not changed.
- `file_path` is not changed.
- Physical scale is not recovered.

So this is not image editing and not physical-size calibration. It is camera-pose phase correction. The output dataset keeps image and mask folders as symlinks to the original processed dataset, while writing new `transforms.json` files.


In [ ]:
import json
import runpy
import shlex
import sys
from pathlib import Path

PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', '/data/abelde/projects/active/Shell_Gaussian'))

# Set this to True when you want to regenerate the dataset.
RUN_TURNTABLE_CANONICALIZATION = False

FOOTSHELL_ROOT = PROJECT_ROOT / 'FootShellGaussian'
GSHELL_ENV = PROJECT_ROOT / 'baselines' / 'GShell' / 'GShell_env'

CANON_INPUT_ROOT = Path('/data/abelde/datasets/processed/gshell_shoes')
CANON_OUTPUT_ROOT = Path('/data/abelde/datasets/processed/gshell_shoes_turntable_canonical')

# Leave this empty to canonicalize every shoe in CANON_INPUT_ROOT.
# Add exact scene folder names if you only want a subset.
CANON_SCENE_NAMES = []

CANON_REFERENCE_FRAME = 'img01.jpg'
CANON_TARGET_ANGLE_DEG = 90
CANON_OVERWRITE = True
CANON_DRY_RUN = False

canonicalize_script = FOOTSHELL_ROOT / 'dataset' / 'canonicalize_gshell_turntable_phase.py'
canonicalize_argv = [
    str(canonicalize_script),
    '--input-root', str(CANON_INPUT_ROOT),
    '--output-root', str(CANON_OUTPUT_ROOT),
    '--reference-frame', CANON_REFERENCE_FRAME,
    '--target-angle-deg', str(CANON_TARGET_ANGLE_DEG),
]
for scene_name in CANON_SCENE_NAMES:
    canonicalize_argv += ['--scene', scene_name]
if CANON_OVERWRITE:
    canonicalize_argv += ['--overwrite']
if CANON_DRY_RUN:
    canonicalize_argv += ['--dry-run']

shell_equivalent = [str(GSHELL_ENV / 'bin' / 'python')] + canonicalize_argv
print(' '.join(shlex.quote(part) for part in shell_equivalent))
print('canonical output root:', CANON_OUTPUT_ROOT)

if RUN_TURNTABLE_CANONICALIZATION:
    old_argv = sys.argv[:]
    try:
        sys.argv = canonicalize_argv
        runpy.run_path(str(canonicalize_script), run_name='__main__')
    finally:
        sys.argv = old_argv
else:
    print('Set RUN_TURNTABLE_CANONICALIZATION = True, then execute this cell to run Section 1.')

summary_path = CANON_OUTPUT_ROOT / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    rows = summary.get('rows', summary if isinstance(summary, list) else [])
    print('summary:', summary_path)
    print('row_count:', len(rows))
    if isinstance(summary, dict):
        for key in [
            'scene_count',
            'status_counts',
            'max_target_error_deg',
            'all_rotations_passed',
            'all_frame_counts_36',
            'original_dataset_modified',
        ]:
            if key in summary:
                print(f'{key}:', summary[key])


## Section 2: Train GShell Meshes

This step trains GShell using the turntable-canonical dataset from Section 1.

Input dataset:

```text
/data/abelde/datasets/processed/gshell_shoes_turntable_canonical
```

Output meshes:

```text
/data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/turntable-512-768
```

The important training settings are:

- config: `shoes_mc_normfix_512_768.json`
- output suffix: `_turntable`
- `SKIP_EXISTING=1`, so already completed shoes are skipped

This cell launches the existing GShell tmux training script. It does not run by default because this is a long GPU job.


In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', '/data/abelde/projects/active/Shell_Gaussian'))
FOOTSHELL_ROOT = PROJECT_ROOT / 'FootShellGaussian'
GSHELL_ROOT = PROJECT_ROOT / 'baselines' / 'GShell'
GSHELL_OUTPUT_ROOT = GSHELL_ROOT / 'output'

# Safety guard: set this to True when you want to launch the tmux training job.
RUN_GSHELL_TRAINING = False

TRAIN_SHOE_NAMES = [
    'Adidas-Yeezy-Boost-350-V2-Desert-Sage-Infant',
    'Adidas-Yeezy-Boost-350-V2-Static-Non-Reflective-Infants',
    'Adidas-Yeezy-Boost-350-V2-Static-Non-Reflective-Kids',
    'Air-Jordan-1-Mid-Wear-Away-Chicago-Gs',
    'Air-Jordan-1-Retro-High-Hyper-Royal-Smoke-Grey-Gs',
    'Air-Jordan-1-Retro-High-Og-Washed-Black-Gs',
    'Air-Jordan-1-Retro-High-Og-White-Cement-Gs',
    'Air-Jordan-12-Retro-Arctic-Punch-Gs',
    'Air-Jordan-13-Retro-Houndstooth-Gs',
    'Air-Jordan-5-Retro-Plaid-Gs',
    'Air-Jordan-6-Retro-Washed-Denim-2022-Gs',
    'Birkenstock-Boston-Suede-Stone-Coin',
    'Crocs-Classic-Clog-Cinnamon-Toast-Crunch',
    'Crocs-Classic-Clog-Cinnamon-Toast-Crunch-Gs',
    'Crocs-Classic-Clog-Cocoa-Puffs-Kids',
    'Crocs-Classic-Clog-Staple-Sidewalk-Luxe',
    'Nike-Calm-Slide-Cinnamon-Monarch',
    'Nike-Cortez-Se-Suede-Pacific-Moss-Infinite-Gold-Muslin-Sail',
    'Ugg-Bailey-Bow-Ii-Boot-Ribbon-Red-Kids',
    'Ugg-Classic-Short-Ii-Boot-Rock-Rose-Toddler',
]

# To train every scene in the dataset, set TRAIN_SHOE_NAMES = [].
TRAIN_SESSION_NAME = 'gshell_turntable_20'
TRAIN_DATASET_ROOT = Path('/data/abelde/datasets/processed/gshell_shoes_turntable_canonical')
TRAIN_CONFIG = GSHELL_ROOT / 'configs' / 'shoes_mc_normfix_512_768.json'
TRAIN_OUTPUT_ROOT = GSHELL_OUTPUT_ROOT / 'turntable-512-768'
TRAIN_SCRIPT = GSHELL_ROOT / 'scripts' / 'train_all_shoes_tmux.sh'

train_env = os.environ.copy()
train_env.update({
    'MIN_FREE_MB': '51200',
    'MAX_PARALLEL_JOBS': '5',
    'SKIP_EXISTING': '1',
    'GSHELL_DATASET_ROOT': str(TRAIN_DATASET_ROOT),
    'GSHELL_CONFIG': str(TRAIN_CONFIG),
    'GSHELL_OUT_SUFFIX': '_turntable',
    'GSHELL_OUTPUT_ROOT': str(TRAIN_OUTPUT_ROOT),
})
train_cmd = ['bash', str(TRAIN_SCRIPT), TRAIN_SESSION_NAME] + TRAIN_SHOE_NAMES

env_prefix = ' '.join(f'{key}={shlex.quote(train_env[key])}' for key in [
    'MIN_FREE_MB', 'MAX_PARALLEL_JOBS', 'SKIP_EXISTING', 'GSHELL_DATASET_ROOT',
    'GSHELL_CONFIG', 'GSHELL_OUT_SUFFIX', 'GSHELL_OUTPUT_ROOT'
])
print('cd', GSHELL_ROOT)
print(env_prefix + ' ' + ' '.join(shlex.quote(part) for part in train_cmd))
print('tmux attach command:', f'tmux attach -t {TRAIN_SESSION_NAME}')

if RUN_GSHELL_TRAINING:
    subprocess.run(train_cmd, cwd=str(GSHELL_ROOT), env=train_env, check=True)
else:
    print('Set RUN_GSHELL_TRAINING = True, then execute this cell to launch training.')

mesh_count = len(list(TRAIN_OUTPUT_ROOT.glob('*/mesh/mesh.obj')))
watertight_count = len(list(TRAIN_OUTPUT_ROOT.glob('*/mesh_watertight/mesh.obj')))
print('existing open mesh count:', mesh_count)
print('existing watertight mesh count:', watertight_count)


## Section 3: Generate Baseline Foot Alignment Debug Outputs

This step places the neutral SUPR foot into each trained shoe mesh using the current basic alignment logic.

It uses the exported meshes from Section 2:

```text
/data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/turntable-512-768/<shoe>_turntable/mesh/mesh.obj
/data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/turntable-512-768/<shoe>_turntable/mesh_watertight/mesh.obj
```

It writes debug outputs here:

```text
/data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/foot_alignment_turntable-512-768
```

This is the reproducible starting placement. The optimizer later uses this as the starting point.


In [ ]:
import os
import runpy
import shlex
import sys
from pathlib import Path

PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', '/data/abelde/projects/active/Shell_Gaussian'))
FOOTSHELL_ROOT = PROJECT_ROOT / 'FootShellGaussian'
GSHELL_ROOT = PROJECT_ROOT / 'baselines' / 'GShell'
GSHELL_ENV = GSHELL_ROOT / 'GShell_env'
GSHELL_OUTPUT_ROOT = GSHELL_ROOT / 'output'

# Safety guard: set this to True when you want to regenerate baseline alignment outputs.
RUN_BASELINE_ALIGNMENT = False

BASELINE_ALIGN_CUDA_VISIBLE_DEVICES = '0'
ALIGN_SHOE_NAMES = list(globals().get('TRAIN_SHOE_NAMES', []))
ALIGN_OVERWRITE = True

ALIGN_DATASET_ROOT = Path('/data/abelde/datasets/processed/gshell_shoes_turntable_canonical')
ALIGN_OUTPUT_ROOT = GSHELL_OUTPUT_ROOT / 'foot_alignment_turntable-512-768'
ALIGN_SCRIPT = FOOTSHELL_ROOT / 'scripts' / 'prepare_dataset_foot_alignment_debug.py'

baseline_align_argv = [
    str(ALIGN_SCRIPT),
    '--dataset-root', str(ALIGN_DATASET_ROOT),
    '--baseline-output-root', str(GSHELL_OUTPUT_ROOT),
    '--baseline-subdir', 'turntable-512-768',
    '--baseline-suffix', '_turntable',
    '--gshell-config', str(GSHELL_ROOT / 'configs' / 'shoes_mc_normfix_512_768.json'),
    '--out-root', str(ALIGN_OUTPUT_ROOT),
    '--device', 'cuda',
]
for shoe_name in ALIGN_SHOE_NAMES:
    baseline_align_argv += ['--shoe-name', shoe_name]
if ALIGN_OVERWRITE:
    baseline_align_argv += ['--overwrite']

shell_equivalent = [str(GSHELL_ENV / 'bin' / 'python')] + baseline_align_argv
print('CUDA_VISIBLE_DEVICES=' + BASELINE_ALIGN_CUDA_VISIBLE_DEVICES, ' '.join(shlex.quote(part) for part in shell_equivalent))
print('baseline alignment output root:', ALIGN_OUTPUT_ROOT)

if RUN_BASELINE_ALIGNMENT:
    old_argv = sys.argv[:]
    old_cuda = os.environ.get('CUDA_VISIBLE_DEVICES')
    try:
        os.environ['CUDA_VISIBLE_DEVICES'] = BASELINE_ALIGN_CUDA_VISIBLE_DEVICES
        sys.argv = baseline_align_argv
        runpy.run_path(str(ALIGN_SCRIPT), run_name='__main__')
    finally:
        sys.argv = old_argv
        if old_cuda is None:
            os.environ.pop('CUDA_VISIBLE_DEVICES', None)
        else:
            os.environ['CUDA_VISIBLE_DEVICES'] = old_cuda
else:
    print('Set RUN_BASELINE_ALIGNMENT = True, then execute this cell to run baseline alignment.')

print('summary csv:', ALIGN_OUTPUT_ROOT / 'summary.csv')
print('contact sheet:', ALIGN_OUTPUT_ROOT / 'all_shoes_contact_sheet.png')


## Section 4: Extract Final 1D Support Footprint And Pseudo-Footbed

This step builds the geometry signals used by the final foot optimizer.

It does not move the foot. It extracts:

- bottom footprint mask
- shoe centerline
- left and right width boundaries
- heel, ball, toe regions
- stable 1D-derived pseudo-footbed heightmap

Input meshes:

```text
/data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/turntable-512-768
```

Output support data:

```text
/data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/final/support_1d
```

The optimizer mainly uses `support_footprint.json` and `pseudo_footbed_heightmap.npz` from this step.


In [ ]:
import json
import runpy
import shlex
import sys
from pathlib import Path

PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', '/data/abelde/projects/active/Shell_Gaussian'))
FOOTSHELL_ROOT = PROJECT_ROOT / 'FootShellGaussian'
GSHELL_ROOT = PROJECT_ROOT / 'baselines' / 'GShell'
GSHELL_ENV = GSHELL_ROOT / 'GShell_env'
GSHELL_OUTPUT_ROOT = GSHELL_ROOT / 'output'

# Safety guard: set this to True when you want to regenerate final support-footbed artifacts.
RUN_SUPPORT_FINAL_1D = False

SUPPORT_MESH_ROOT = GSHELL_OUTPUT_ROOT / 'turntable-512-768'
SUPPORT_OUTPUT_ROOT = GSHELL_OUTPUT_ROOT / 'final' / 'support_1d'
SUPPORT_SCENE_NAMES = []
SUPPORT_OVERWRITE = True

SUPPORT_GRID_RESOLUTION = 192
SUPPORT_FOOTBED_OFFSET = 0.015
SUPPORT_HEIGHTMAP_MIN_SAMPLES_PER_CELL = 2
SUPPORT_HEIGHTMAP_SMOOTH_SIGMA = 1.25
SUPPORT_HEIGHTMAP_PROFILE_CLIP = 0.025
SUPPORT_FOOTBED_INNER_MARGIN_CELLS = 7
SUPPORT_SMOOTH_FOOTBED_WINDOW_FRACTION = 0.18
SUPPORT_FOOTBED_HEIGHT_FRACTION = 0.22
SUPPORT_OPEN_BOUNDARY_FOOTBED_OFFSET = None
SUPPORT_OPEN_BOUNDARY_FOOTBED_OFFSET_RATIO = 0.055
SUPPORT_OPEN_BOUNDARY_FOOTBED_OFFSET_MIN = 0.008
SUPPORT_OPEN_BOUNDARY_FOOTBED_OFFSET_MAX = 0.022

support_script = FOOTSHELL_ROOT / 'scripts' / 'run_support_footbed_analysis.py'
support_argv = [
    str(support_script),
    '--mesh-root', str(SUPPORT_MESH_ROOT),
    '--output-root', str(SUPPORT_OUTPUT_ROOT),
    '--grid-resolution', str(SUPPORT_GRID_RESOLUTION),
    '--footbed-offset', str(SUPPORT_FOOTBED_OFFSET),
    '--heightmap-min-samples-per-cell', str(SUPPORT_HEIGHTMAP_MIN_SAMPLES_PER_CELL),
    '--heightmap-smooth-sigma', str(SUPPORT_HEIGHTMAP_SMOOTH_SIGMA),
    '--heightmap-profile-clip', str(SUPPORT_HEIGHTMAP_PROFILE_CLIP),
    '--footbed-inner-margin-cells', str(SUPPORT_FOOTBED_INNER_MARGIN_CELLS),
    '--smooth-footbed-window-fraction', str(SUPPORT_SMOOTH_FOOTBED_WINDOW_FRACTION),
    '--footbed-height-fraction', str(SUPPORT_FOOTBED_HEIGHT_FRACTION),
    '--open-boundary-footbed-offset-ratio', str(SUPPORT_OPEN_BOUNDARY_FOOTBED_OFFSET_RATIO),
    '--open-boundary-footbed-offset-min', str(SUPPORT_OPEN_BOUNDARY_FOOTBED_OFFSET_MIN),
    '--open-boundary-footbed-offset-max', str(SUPPORT_OPEN_BOUNDARY_FOOTBED_OFFSET_MAX),
]
if SUPPORT_OPEN_BOUNDARY_FOOTBED_OFFSET is not None:
    support_argv += ['--open-boundary-footbed-offset', str(SUPPORT_OPEN_BOUNDARY_FOOTBED_OFFSET)]
for scene_name in SUPPORT_SCENE_NAMES:
    support_argv += ['--scene', scene_name]
if SUPPORT_OVERWRITE:
    support_argv += ['--overwrite']

shell_equivalent = [str(GSHELL_ENV / 'bin' / 'python')] + support_argv
print(' '.join(shlex.quote(part) for part in shell_equivalent))
print('support-footbed output root:', SUPPORT_OUTPUT_ROOT)

if RUN_SUPPORT_FINAL_1D:
    old_argv = sys.argv[:]
    try:
        sys.argv = support_argv
        runpy.run_path(str(support_script), run_name='__main__')
    finally:
        sys.argv = old_argv
else:
    print('Set RUN_SUPPORT_FINAL_1D = True, then execute this cell to run support-footbed extraction.')

summary_path = SUPPORT_OUTPUT_ROOT / 'support_footbed_summary.json'
if summary_path.exists():
    rows = json.loads(summary_path.read_text())
    errors = [row for row in rows if 'error' in row]
    print('summary:', summary_path)
    print('rows:', len(rows))
    print('errors:', len(errors))
    print('npz_count:', len(list(SUPPORT_OUTPUT_ROOT.glob('*/pseudo_footbed_heightmap.npz'))))


## Section 5: Run Final 1D Heightmap Foot Optimization

This is the current reproducible foot placement step.

The shoe mesh stays fixed. The optimizer moves the SUPR foot by changing:

- uniform scale
- yaw
- pitch
- roll
- X/Y/Z translation

It uses:

- baseline alignment from Section 3
- final 1D support-footbed data from Section 4
- no warm start from older optimized folders
- no extra footbed offset

Output optimized foot placements:

```text
/data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/final/foot_alignment_1d_heightmap
```

This is the folder we currently use for debugging foot placement and for the next mSDF logic.


In [1]:
import json
import os
import runpy
import shlex
import sys
from pathlib import Path

PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', '/data/abelde/projects/active/Shell_Gaussian'))
FOOTSHELL_ROOT = PROJECT_ROOT / 'FootShellGaussian'
GSHELL_ROOT = PROJECT_ROOT / 'baselines' / 'GShell'
GSHELL_ENV = GSHELL_ROOT / 'GShell_env'
GSHELL_OUTPUT_ROOT = GSHELL_ROOT / 'output'

# Safety guard: set this to True when you want to run the final optimizer.
RUN_FINAL_1D_OPTIMIZER = True

OPT_CUDA_VISIBLE_DEVICES = '5'
OPT_SCENE_NAMES = []
# Smoke-test example:
# OPT_SCENE_NAMES = [
#     'Adidas-Yeezy-Boost-350-V2-Desert-Sage-Infant_turntable',
#     'Birkenstock-Boston-Suede-Stone-Coin_turntable',
#     'Crocs-Classic-Clog-Cinnamon-Toast-Crunch-Gs_turntable',
#     'Ugg-Classic-Short-Ii-Boot-Rock-Rose-Toddler_turntable',
# ]
OPT_OVERWRITE = True

OPT_MESH_ROOT = GSHELL_OUTPUT_ROOT / 'turntable-512-768'
OPT_SUPPORT_ROOT = GSHELL_OUTPUT_ROOT / 'final' / 'support_1d'
OPT_BASELINE_ALIGNMENT_ROOT = GSHELL_OUTPUT_ROOT / 'foot_alignment_turntable-512-768'
OPT_OUTPUT_ROOT = GSHELL_OUTPUT_ROOT / 'final' / 'foot_alignment_1d_heightmap'
OPT_FOOT_OBJ = PROJECT_ROOT / 'baselines' / 'SUPR' / 'output' / 'debug_playground' / 'supr_male_right_foot_neutral.obj'
OPT_SCRIPT = FOOTSHELL_ROOT / 'scripts' / 'run_foot_fit_optimization.py'

optimizer_argv = [
    str(OPT_SCRIPT),
    '--fit-version', 'final_1d',
    '--mesh-root', str(OPT_MESH_ROOT),
    '--support-root', str(OPT_SUPPORT_ROOT),
    '--baseline-alignment-root', str(OPT_BASELINE_ALIGNMENT_ROOT),
    '--output-root', str(OPT_OUTPUT_ROOT),
    '--foot-obj', str(OPT_FOOT_OBJ),
    '--device', 'cuda',
    '--style-mode', 'auto',
    '--adam-steps', '160',
    '--lbfgs-steps', '25',
]
for scene_name in OPT_SCENE_NAMES:
    optimizer_argv += ['--scene', scene_name]
if OPT_OVERWRITE:
    optimizer_argv += ['--overwrite']

shell_equivalent = [str(GSHELL_ENV / 'bin' / 'python')] + optimizer_argv
print('CUDA_VISIBLE_DEVICES=' + OPT_CUDA_VISIBLE_DEVICES, ' '.join(shlex.quote(part) for part in shell_equivalent))
print('optimizer output root:', OPT_OUTPUT_ROOT)

if RUN_FINAL_1D_OPTIMIZER:
    old_argv = sys.argv[:]
    old_cuda = os.environ.get('CUDA_VISIBLE_DEVICES')
    try:
        os.environ['CUDA_VISIBLE_DEVICES'] = OPT_CUDA_VISIBLE_DEVICES
        sys.argv = optimizer_argv
        runpy.run_path(str(OPT_SCRIPT), run_name='__main__')
    finally:
        sys.argv = old_argv
        if old_cuda is None:
            os.environ.pop('CUDA_VISIBLE_DEVICES', None)
        else:
            os.environ['CUDA_VISIBLE_DEVICES'] = old_cuda
else:
    print('Set RUN_FINAL_1D_OPTIMIZER = True, then execute this cell to run final foot optimization.')

summary_path = OPT_OUTPUT_ROOT / 'summary.json'
if summary_path.exists():
    rows = json.loads(summary_path.read_text())
    ok = [row for row in rows if row.get('status') == 'ok']
    print('summary:', summary_path)
    print('rows:', len(rows))
    print('ok:', len(ok))
    print('errors:', len(rows) - len(ok))
    print('fit_metrics:', len(list(OPT_OUTPUT_ROOT.glob('*/fit_metrics.json'))))
    print('contact_sheets:', len(list(OPT_OUTPUT_ROOT.glob('*/fit_contact_sheet.png'))))
    print('dataset contact sheet:', OPT_OUTPUT_ROOT / 'all_shoes_optimized_contact_sheet.png')


CUDA_VISIBLE_DEVICES=5 /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/bin/python /data/abelde/projects/active/Shell_Gaussian/FootShellGaussian/scripts/run_foot_fit_optimization.py --fit-version final_1d --mesh-root /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/turntable-512-768 --support-root /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/final/support_1d --baseline-alignment-root /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/foot_alignment_turntable-512-768 --output-root /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/final/foot_alignment_1d_heightmap --foot-obj /data/abelde/projects/active/Shell_Gaussian/baselines/SUPR/output/debug_playground/supr_male_right_foot_neutral.obj --device cuda --style-mode auto --adam-steps 160 --lbfgs-steps 25 --overwrite
optimizer output root: /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/final/foot_alignment_1d_heightmap
[

KeyboardInterrupt: 